In [6]:
#
#  notebooks/01_define_persona_vector.ipynb
#
# in terminal export PPT_MODEL_ID="meta-llama/Meta-Llama-3.2-1B-Instruct"  # or ...-3B-Instruct - waiting authorization from meta
# export PPT_MODEL_ID="TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# (alternatives)
# export PPT_MODEL_ID="Qwen/Qwen2.5-1.5B-Instruct"
# export PPT_MODEL_ID="Qwen/Qwen2.5-0.5B-Instruct"
# export PPT_MODEL_ID="google/gemma-2-2b-it" - try this Sept 25
# =============================================================================
# Cell 1: Setup and Imports
# =============================================================================
import torch
import json
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.notebook import tqdm

# --- Configuration ---
# The base model you will fine-tune.

#MODEL_ID = os.getenv("PPT_MODEL_ID", "google/gemma-2-2b-it")
MODEL_ID = os.getenv("PPT_MODEL_ID", "meta-llama/Llama-3.2-1B-Instruct")

#MODEL_ID = "google/gemma-2b-it" 
#MODEL_ID = "meta-llama/Meta-Llama-3.2-8B-Instruct" 
#meta-llama/Meta-Llama-3.2-1B-Instruct
#meta-llama/Meta-Llama-3.2-3B-Instruct 

# Path to the file with contrasting text pairs.
CONTRASTING_PAIRS_PATH = "../data/persona_vector_probes/contrasting_pairs.jsonl"

# The layer from which to extract activations. -2 is a common choice (second-to-last layer).
LAYER_TO_EXTRACT = -2 

# Path to save the final vector.
VECTOR_OUTPUT_PATH = "../vectors/cautious_scientist_vector_Llama-3.2-1B.pt"
# ---------------------


In [2]:
#from huggingface_hub import login
#login(token="...")
#or
#try:
#    from dotenv import load_dotenv; load_dotenv()
#except Exception:
#    pass


In [8]:

# =============================================================================
# Cell 2: Load Base Model and Tokenizer
# =============================================================================
print(f"Loading base model: {MODEL_ID}")
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

use_cuda = torch.cuda.is_available()
dtype = torch.bfloat16 if use_cuda else torch.float32  # FP32 on CPU

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=dtype,                   
    device_map=("auto" if use_cuda else None),
    low_cpu_mem_usage=True,        # helps on CPU
)
if not use_cuda:
    model = model.to("cpu")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.eval()
print("Loaded on", next(model.parameters()).device, "| dtype:", dtype)



Loading base model: meta-llama/Llama-3.2-1B-Instruct
Loaded on cpu | dtype: torch.float32


In [9]:
# =============================================================================
# Cell 3: Activation Extraction Function
# =============================================================================
import torch

# (optional) if CPU feels sluggish, cap PyTorch threads a bit
# torch.set_num_threads(8)

def get_mean_activations(text: str, layer_idx: int):
    """
    Encode `text` and return the mean activation vector from the chosen layer.
    Works with negative indices (e.g., -2 = second-to-last).
    """
    device = next(model.parameters()).device
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.inference_mode():
        out = model(**inputs, output_hidden_states=True)

    hs = out.hidden_states
    n_layers = len(hs)
    idx = layer_idx if layer_idx >= 0 else n_layers + layer_idx
    vec = hs[idx].mean(dim=1).squeeze(0).detach().cpu()  # (hidden_dim,)
    return vec

# quick smoke test
test_vec = get_mean_activations("hello world", LAYER_TO_EXTRACT)
print("Test vector:", tuple(test_vec.shape), "| norm:", float(test_vec.norm()))





Test vector: (2048,) | norm: 267.31402587890625


In [10]:
# =============================================================================
# Cell 4: Build persona vector from contrasting pairs (safe tqdm)
# =============================================================================
import json, torch
from pathlib import Path
try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs): return x

pairs = []
with open(CONTRASTING_PAIRS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        pairs.append(json.loads(line))

# (Optional) quick run first
# pairs = pairs[:20]

pos_vecs, neg_vecs = [], []
for ex in tqdm(pairs, desc="Computing activations"):
    pos_vecs.append(get_mean_activations(ex["positive"], LAYER_TO_EXTRACT))
    neg_vecs.append(get_mean_activations(ex["negative"], LAYER_TO_EXTRACT))

mean_pos = torch.stack(pos_vecs).mean(dim=0)
mean_neg = torch.stack(neg_vecs).mean(dim=0)

persona_vector = mean_pos - mean_neg
persona_vector = persona_vector / persona_vector.norm()

print("persona_vector shape:", tuple(persona_vector.shape), " | norm:", float(persona_vector.norm()))




Computing activations:   0%|          | 0/70 [00:00<?, ?it/s]

persona_vector shape: (2048,)  | norm: 0.9999998807907104


In [11]:
# =============================================================================
# Cell 5: Save vector
# =============================================================================
Path("../vectors").mkdir(parents=True, exist_ok=True)
torch.save(persona_vector, VECTOR_OUTPUT_PATH)
print("Saved to:", VECTOR_OUTPUT_PATH)

Saved to: ../vectors/cautious_scientist_vector_Llama-3.2-1B.pt
